In [1]:
!pip install kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 753.8 kB/s eta 0:00:0031m1.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 kB 1.7 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 1.9 MB/s eta 0:00:002.7 MB/s eta 0:00:01


In [3]:
import pandas as pd

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("asaniczka/top-spotify-songs-in-73-countries-daily-updated")

print("Path to dataset files:", path)

100%|████████████████████████████████████████| 162M/162M [01:08<00:00, 2.49MB/s]

Extracting files...


Path to dataset files: /home/marija/.cache/kagglehub/datasets/asaniczka/top-spotify-songs-in-73-countries-daily-updated/versions/608


In [5]:
import os
print("Fajlovi u datasetu:", os.listdir(path))
csv_path = os.path.join(path, 'universal_top_spotify_songs.csv')

Fajlovi u datasetu: ['universal_top_spotify_songs.csv']


In [6]:
df = pd.read_csv(csv_path)

In [7]:
df.head()

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,2RkZ5LkEzeHGRsmDqKwmaJ,Ordinary,Alex Warren,1,1,0,NaN,2025-06-11,95,False,...,2,-6.141,1,0.0600,0.704000,0.000007,0.0550,0.391,168.115,3
1,42UBPzRMh5yyz0EDPr6fr1,Manchild,Sabrina Carpenter,2,-1,48,NaN,2025-06-11,89,True,...,7,-5.087,1,0.0572,0.122000,0.000000,0.3170,0.811,123.010,4
2,0FTmksd2dxiE5e3rWyJXs6,back to friends,sombr,3,0,1,NaN,2025-06-11,98,False,...,1,-2.291,1,0.0301,0.000094,0.000088,0.0929,0.235,92.855,4
3,7so0lgd0zP2Sbgs2d7a1SZ,Die With A Smile,"Lady Gaga, Bruno Mars",4,0,-1,NaN,2025-06-11,91,False,...,6,-7.727,0,0.0317,0.289000,0.000000,0.1260,0.498,157.964,3
4,6dOtVTDdiauQNBQEDOtlAB,BIRDS OF A FEATHER,Billie Eilish,5,1,0,NaN,2025-06-11,100,False,...,2,-10.171,1,0.0358,0.200000,0.060800,0.1170,0.438,104.978,4


In [8]:
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])

In [9]:
df = df[df["snapshot_date"] <= "2025-03-07"]

In [10]:
df = df.reset_index(drop=True)

In [11]:
print(df.head())
print(len(df))

               spotify_id                name                artists  \
0  2plbrEY59IikOBgBGLjaoe    Die With A Smile  Lady Gaga, Bruno Mars   
1  2CGNAOSuO1MEFCbBRgUzjd   luther (with sza)    Kendrick Lamar, SZA   
2  4wJ5Qq0jBN4ajy7ouZIV1c                APT.       ROSÉ, Bruno Mars   
3  6dOtVTDdiauQNBQEDOtlAB  BIRDS OF A FEATHER          Billie Eilish   
4  6AI3ezQ4o3HUoP6Dhudph3         Not Like Us         Kendrick Lamar   

   daily_rank  daily_movement  weekly_movement country snapshot_date  \
0           1               0                0     NaN    2025-03-07   
1           2               0                0     NaN    2025-03-07   
2           3               0                0     NaN    2025-03-07   
3           4               0                0     NaN    2025-03-07   
4           5               0                0     NaN    2025-03-07   

   popularity  is_explicit  ...  key loudness mode  speechiness  acousticness  \
0          96        False  ...    6   -7.777    0   

In [12]:
df.to_csv('universal_top_spotify_songs.csv', index=False)

---
## Drugi formati za čuvanje podataka: pickle i parquet

Iznad su podaci sačuvani u CSV. CSV je čitljiv u svakom alatu, ali ne pamti tipove kolona
i zauzima najviše prostora. Ispod su još dva formata: **pickle** (binarni Python format)
i **parquet** (binarni kolonarni format).

Za parquet je potrebna biblioteka `pyarrow`, za rad sa bazom `duckdb`, a za Spark `pyspark`
(traži Javu 17 ili 21 - ovde je instalirana Java 21, pa ide pyspark 4.x; verzija 3.5 ne podržava Javu 21).

In [1]:
#!pip install pyarrow duckdb pyspark

### Pickle

`to_pickle` serijalizuje DataFrame tačno onakav kakav jeste - zajedno sa indeksom i tipovima kolona.
Čitanje je brzo, ali fajl može da pročita samo Python i vezan je za verzije biblioteka.

In [14]:
df.to_pickle('universal_top_spotify_songs.pkl')

df_pkl = pd.read_pickle('universal_top_spotify_songs.pkl')
print(df_pkl.shape)
print(df_pkl['snapshot_date'].dtype)   # tip je sačuvan

(1793225, 25)
datetime64[ns]


### Parquet

Parquet je **kolonarni** format: vrednosti se na disku čuvaju po kolonama, a ne po redovima.
Zbog toga se dobro komprimuje i omogucava da se pročita samo podskup kolona, bez čitanja celog fajla.
Šemu (imena i tipove kolona) nosi u sebi, pa se tipovi ne gube kao kod CSV-a.

Pandas podrazumevano upisuje vremenske oznake u nanosekundama, a Spark takav zapis ne ume da
pročita, pa se kod upisa traži preciznost od mikrosekunde (`coerce_timestamps='us'`). Ovo je tipican
primer da binarni formati imaju varijante, pa treba voditi računa ko će fajl kasnije citati.

In [15]:
df.to_parquet('universal_top_spotify_songs.parquet',
              engine='pyarrow',
              index=False,
              coerce_timestamps='us',        # vremenske oznake u mikrosekundama...
              allow_truncated_timestamps=True)  # ...jer Spark ne čita nanosekunde

df_parquet = pd.read_parquet('universal_top_spotify_songs.parquet')
print(df_parquet.shape)
print(df_parquet['snapshot_date'].dtype)

(1793225, 25)
datetime64[us]


In [16]:
# prednost kolonarnog formata: učitavanje samo potrebnih kolona
df_dve_kolone = pd.read_parquet('universal_top_spotify_songs.parquet',
                                columns=['country', 'danceability'])
print(df_dve_kolone.shape)
df_dve_kolone.head()

(1793225, 2)


,country,danceability
0,None,0.521
1,None,0.707
2,None,0.777
3,None,0.747
4,None,0.898


### Poređenje tri formata

Veličina fajla na disku i vreme potrebno da se podaci učitaju nazad u pandas.

In [17]:
import os, time

fajlovi = {
    'CSV':     'universal_top_spotify_songs.csv',
    'pickle':  'universal_top_spotify_songs.pkl',
    'parquet': 'universal_top_spotify_songs.parquet',
}
citaci = {'CSV': pd.read_csv, 'pickle': pd.read_pickle, 'parquet': pd.read_parquet}

print(f"{'format':<10}{'veličina (MB)':>15}{'vreme čitanja (s)':>20}")
for naziv, putanja in fajlovi.items():
    mb = os.path.getsize(putanja) / 1024**2
    pocetak = time.perf_counter()
    citaci[naziv](putanja)
    trajanje = time.perf_counter() - pocetak
    print(f"{naziv:<10}{mb:>15.1f}{trajanje:>20.2f}")

format      veličina (MB)   vreme čitanja (s)


CSV                 320.2                4.30


pickle              306.9                0.34


parquet              45.3                0.42


In [18]:
# CSV ne pamti tipove - datum se vraća kao tekst; parquet i pickle ga čuvaju
print("original:", df['snapshot_date'].dtype)
print("iz CSV-a:", pd.read_csv('universal_top_spotify_songs.csv', nrows=5)['snapshot_date'].dtype)
print("iz parquet-a:", pd.read_parquet('universal_top_spotify_songs.parquet',
                                       columns=['snapshot_date']).dtypes.iloc[0])
print("iz pickle-a:", pd.read_pickle('universal_top_spotify_songs.pkl')['snapshot_date'].dtype)

original: datetime64[ns]
iz CSV-a: object
iz parquet-a: datetime64[us]


iz pickle-a: datetime64[ns]


---
## DBeaver

DBeaver je grafički SQL klijent - njemu je potrebna baza na koju se kači. Zato se podaci
prvo upisuju u **DuckDB** bazu (jedan fajl na disku, bez servera), a onda se ta baza otvara
u DBeaver-u i nad njom se pišu obični SQL upiti.

In [19]:
import duckdb

con = duckdb.connect('spotify.duckdb')
con.register('df_view', df)
con.execute("CREATE OR REPLACE TABLE songs AS SELECT * FROM df_view")

print(con.sql("SELECT COUNT(*) AS broj_redova FROM songs").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   broj_redova
0      1793225


### Kačenje DBeaver-a na ovu bazu

1. U DBeaver-u: **Database -> New Database Connection**
2. Izabrati **DuckDB** i kliknuti Next (DBeaver sam skida drajver ako ga nema)
3. U polje **Path** upisati punu putanju do fajla: `/home/marija/Desktop/IP2/spotify.duckdb`
4. **Test Connection**, pa **Finish**
5. Tabela `songs` se vidi u stablu levo, a upiti se kucaju u SQL editoru (`Ctrl+]` za novi editor,
   `Ctrl+Enter` za izvršavanje)

Dve napomene:
- DuckDB dozvoljava **samo jednog pisca** nad fajlom. Ako je Python konekcija još otvorena,
  DBeaver neće moći da se zakaci - treba prvo pozvati `con.close()` (poslednja ćelija ove sekcije).
- Ako DBeaver prijavi neslaganje verzije fajla, drajver se ažurira kroz
  **Database -> Driver Manager -> DuckDB -> Edit -> Libraries -> Download/Update**.

Upiti ispod su isti oni koji se kucaju u DBeaver-u; ovde se izvršavaju iz Pythona da bi se
u notebook-u video rezultat.

#### Primer 1: SELECT sa WHERE i ORDER BY

Top 10 pesama u SAD na dan 2025-03-07.

```sql
SELECT daily_rank, name, artists, popularity
FROM songs
WHERE country = 'US' AND snapshot_date = DATE '2025-03-07'
ORDER BY daily_rank
LIMIT 10;
```

In [20]:
con.sql("""
    SELECT daily_rank, name, artists, popularity
    FROM songs
    WHERE country = 'US' AND snapshot_date = DATE '2025-03-07'
    ORDER BY daily_rank
    LIMIT 10
""").df()

,daily_rank,name,artists,popularity
0,1,luther (with sza),"Kendrick Lamar, SZA",89
1,2,tv off (feat. lefty gunplay),"Kendrick Lamar, Lefty Gunplay",90
2,3,DENIAL IS A RIVER,Doechii,88
3,4,Not Like Us,Kendrick Lamar,92
4,5,NOKIA,Drake,83
5,6,Die With A Smile,"Lady Gaga, Bruno Mars",96
6,7,Sports car,Tate McRae,77
7,8,All The Stars (with SZA),"Kendrick Lamar, SZA",90
8,9,Anxiety,Doechii,0
9,10,Who,Jimin,86


#### Primer 2: GROUP BY sa agregacijom i HAVING

Prosečna plesnost i glasnoća po državi, samo za države sa više od 10.000 zapisa.

```sql
SELECT country,
       COUNT(*)                     AS broj_zapisa,
       ROUND(AVG(danceability), 3)  AS prosecna_plesnost,
       ROUND(AVG(loudness), 2)      AS prosecna_glasnoca
FROM songs
WHERE country IS NOT NULL
GROUP BY country
HAVING COUNT(*) > 10000
ORDER BY prosecna_plesnost DESC
LIMIT 10;
```

In [21]:
con.sql("""
    SELECT country,
           COUNT(*)                     AS broj_zapisa,
           ROUND(AVG(danceability), 3)  AS prosecna_plesnost,
           ROUND(AVG(loudness), 2)      AS prosecna_glasnoca
    FROM songs
    WHERE country IS NOT NULL
    GROUP BY country
    HAVING COUNT(*) > 10000
    ORDER BY prosecna_plesnost DESC
    LIMIT 10
""").df()

,country,broj_zapisa,prosecna_plesnost,prosecna_glasnoca
0,EC,24508,0.753,-5.94
1,CO,24514,0.752,-5.76
2,PE,24656,0.747,-5.82
3,CL,24705,0.745,-6.11
4,SV,24762,0.745,-6.08
5,VE,23863,0.741,-6.23
6,RO,24668,0.739,-5.74
7,HN,24762,0.734,-6.21
8,MX,24706,0.732,-6.51
9,NI,24770,0.731,-6.26


#### Primer 3: GROUP BY po izvođaču

Izvođači sa najviše pojavljivanja na dnevnim top listama.

```sql
SELECT artists,
       COUNT(*)                  AS pojavljivanja,
       COUNT(DISTINCT country)   AS broj_drzava,
       MIN(daily_rank)           AS najbolji_plasman
FROM songs
GROUP BY artists
ORDER BY pojavljivanja DESC
LIMIT 10;
```

In [22]:
con.sql("""
    SELECT artists,
           COUNT(*)                  AS pojavljivanja,
           COUNT(DISTINCT country)   AS broj_drzava,
           MIN(daily_rank)           AS najbolji_plasman
    FROM songs
    GROUP BY artists
    ORDER BY pojavljivanja DESC
    LIMIT 10
""").df()

,artists,pojavljivanja,broj_drzava,najbolji_plasman
0,Billie Eilish,27838,67,1
1,Sabrina Carpenter,26725,64,1
2,Bad Bunny,21262,50,1
3,Taylor Swift,18630,64,1
4,KAROL G,18431,26,1
5,Jimin,15564,50,1
6,Benson Boone,14519,54,1
7,Teddy Swims,13364,31,1
8,Feid,12545,18,1
9,"Lady Gaga, Bruno Mars",12096,69,1


#### Primer 4: upit direktno nad parquet fajlom

DuckDB ume da čita parquet bez prethodnog uvoza u bazu - umesto imena tabele navodi se
putanja do fajla. Isti upit radi i u DBeaver-u.

```sql
SELECT COUNT(*)                 AS broj_redova,
       COUNT(DISTINCT country)  AS broj_drzava,
       MIN(snapshot_date)       AS od,
       MAX(snapshot_date)       AS do
FROM 'universal_top_spotify_songs.parquet';
```

In [23]:
con.sql("""
    SELECT COUNT(*)                 AS broj_redova,
           COUNT(DISTINCT country)  AS broj_drzava,
           MIN(snapshot_date)       AS od,
           MAX(snapshot_date)       AS do
    FROM 'universal_top_spotify_songs.parquet'
""").df()

,broj_redova,broj_drzava,od,do
0,1793225,72,2023-10-18,2025-03-07


In [24]:
# zatvaranje konekcije - obavezno pre kačenja DBeaver-a na isti fajl
con.close()
print("Konekcija zatvorena, baza spotify.duckdb je slobodna za DBeaver.")

Konekcija zatvorena, baza spotify.duckdb je slobodna za DBeaver.


---
## Apache Spark

Isti upiti, ali kroz Spark. Ovde se pokreće lokalno (`local[*]` znači: koristi sva jezgra ovog
računara), dok bi u produkciji isti kod išao na klaster. Spark čita **parquet**, jer je šema već
upisana u fajl pa nije potrebno pogađanje tipova kao kod CSV-a.

In [25]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .master('local[*]')
         .appName('spotify')
         .config('spark.driver.memory', '2g')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

sdf = spark.read.parquet('universal_top_spotify_songs.parquet')
print("Broj redova:", sdf.count())
sdf.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/09 01:04:33 WARN Utils: Your hostname, marija-HP-EliteBook-850-G8-Notebook-PC, resolves to a loopback address: 127.0.1.1; using 192.168.1.5 instead (on interface wlp0s20f3)
26/10/09 01:04:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/09 01:04:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Broj redova: 1793225
root
 |-- spotify_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- daily_rank: long (nullable = true)
 |-- daily_movement: long (nullable = true)
 |-- weekly_movement: long (nullable = true)
 |-- country: string (nullable = true)
 |-- snapshot_date: timestamp_ntz (nullable = true)
 |-- popularity: long (nullable = true)
 |-- is_explicit: boolean (nullable = true)
 |-- duration_ms: long (nullable = true)
 |-- album_name: string (nullable = true)
 |-- album_release_date: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: long (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: long (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: doubl

### Varijanta 1: DataFrame API

Spark ima svoj API u kome se upit gradi pozivima metoda. Transformacije su *lenje* - ništa se ne
računa dok se ne pozove akcija kao što je `show()` ili `count()`.

In [26]:
# Primer 1: SELECT sa WHERE i ORDER BY
(sdf
 .filter((F.col('country') == 'US') & (F.col('snapshot_date') == '2025-03-07'))
 .select('daily_rank', 'name', 'artists', 'popularity')
 .orderBy('daily_rank')
 .show(10, truncate=False))

+----------+----------------------------+-----------------------------+----------+
|daily_rank|name                        |artists                      |popularity|
+----------+----------------------------+-----------------------------+----------+
|1         |luther (with sza)           |Kendrick Lamar, SZA          |89        |
|2         |tv off (feat. lefty gunplay)|Kendrick Lamar, Lefty Gunplay|90        |
|3         |DENIAL IS A RIVER           |Doechii                      |88        |
|4         |Not Like Us                 |Kendrick Lamar               |92        |
|5         |NOKIA                       |Drake                        |83        |
|6         |Die With A Smile            |Lady Gaga, Bruno Mars        |96        |
|7         |Sports car                  |Tate McRae                   |77        |
|8         |All The Stars (with SZA)    |Kendrick Lamar, SZA          |90        |
|9         |Anxiety                     |Doechii                      |0         |
|10 

In [27]:
# Primer 2: GROUP BY sa agregacijom i HAVING (u API-ju je to filter posle agregacije)
(sdf
 .filter(F.col('country').isNotNull())
 .groupBy('country')
 .agg(F.count('*').alias('broj_zapisa'),
      F.round(F.avg('danceability'), 3).alias('prosecna_plesnost'),
      F.round(F.avg('loudness'), 2).alias('prosecna_glasnoca'))
 .filter(F.col('broj_zapisa') > 10000)
 .orderBy(F.desc('prosecna_plesnost'))
 .show(10))

+-------+-----------+-----------------+-----------------+
|country|broj_zapisa|prosecna_plesnost|prosecna_glasnoca|
+-------+-----------+-----------------+-----------------+
|     EC|      24508|            0.753|            -5.94|
|     CO|      24514|            0.752|            -5.76|
|     PE|      24656|            0.747|            -5.82|
|     SV|      24762|            0.745|            -6.08|
|     CL|      24705|            0.745|            -6.11|
|     VE|      23863|            0.741|            -6.23|
|     RO|      24668|            0.739|            -5.74|
|     HN|      24762|            0.734|            -6.21|
|     MX|      24706|            0.732|            -6.51|
|     GT|      24758|            0.731|            -6.19|
+-------+-----------+-----------------+-----------------+
only showing top 10 rows


### Varijanta 2: SQL

Nad DataFrame-om se registruje privremeni pogled i onda se piše isti SQL kao u DuckDB-u.
Tekst upita je identičan - razlikuje se samo mehanizam koji ga izvršava.

In [28]:
sdf.createOrReplaceTempView('songs')

spark.sql("""
    SELECT daily_rank, name, artists, popularity
    FROM songs
    WHERE country = 'US' AND snapshot_date = DATE '2025-03-07'
    ORDER BY daily_rank
    LIMIT 10
""").show(truncate=False)

+----------+----------------------------+-----------------------------+----------+
|daily_rank|name                        |artists                      |popularity|
+----------+----------------------------+-----------------------------+----------+
|1         |luther (with sza)           |Kendrick Lamar, SZA          |89        |
|2         |tv off (feat. lefty gunplay)|Kendrick Lamar, Lefty Gunplay|90        |
|3         |DENIAL IS A RIVER           |Doechii                      |88        |
|4         |Not Like Us                 |Kendrick Lamar               |92        |
|5         |NOKIA                       |Drake                        |83        |
|6         |Die With A Smile            |Lady Gaga, Bruno Mars        |96        |
|7         |Sports car                  |Tate McRae                   |77        |
|8         |All The Stars (with SZA)    |Kendrick Lamar, SZA          |90        |
|9         |Anxiety                     |Doechii                      |0         |
|10 

In [29]:
spark.sql("""
    SELECT artists,
           COUNT(*)                  AS pojavljivanja,
           COUNT(DISTINCT country)   AS broj_drzava,
           MIN(daily_rank)           AS najbolji_plasman
    FROM songs
    GROUP BY artists
    ORDER BY pojavljivanja DESC
    LIMIT 10
""").show(truncate=False)

+---------------------+-------------+-----------+----------------+
|artists              |pojavljivanja|broj_drzava|najbolji_plasman|
+---------------------+-------------+-----------+----------------+
|Billie Eilish        |27838        |67         |1               |
|Sabrina Carpenter    |26725        |64         |1               |
|Bad Bunny            |21262        |50         |1               |
|Taylor Swift         |18630        |64         |1               |
|KAROL G              |18431        |26         |1               |
|Jimin                |15564        |50         |1               |
|Benson Boone         |14519        |54         |1               |
|Teddy Swims          |13364        |31         |1               |
|Feid                 |12545        |18         |1               |
|Lady Gaga, Bruno Mars|12096        |69         |1               |
+---------------------+-------------+-----------+----------------+



In [30]:
# plan izvršavanja - Spark pokazuje kako će upit zaista izvršiti
spark.sql("""
    SELECT country, COUNT(*) AS broj_zapisa
    FROM songs
    WHERE country IS NOT NULL
    GROUP BY country
""").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[country#6], functions=[count(1)])
   +- Exchange hashpartitioning(country#6, 200), ENSURE_REQUIREMENTS, [plan_id=272]
      +- HashAggregate(keys=[country#6], functions=[partial_count(1)])
         +- Filter isnotnull(country#6)
            +- FileScan parquet [country#6] Batched: true, DataFilters: [isnotnull(country#6)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/marija/Desktop/IP2/universal_top_spotify_songs.parquet], PartitionFilters: [], PushedFilters: [IsNotNull(country)], ReadSchema: struct<country:string>




In [31]:
spark.stop()
print("Spark sesija zatvorena.")

Spark sesija zatvorena.
